# ViDoRe V3 Finance EN — Short QARF/QPAF Oracle

This notebook runs the exploratory **Global Fusion → QARF → QPAF** oracle on all
309 English queries and the full 2,942-page `vidore_v3_finance_en` corpus.

- Dataset revision is pinned.
- Qrels are used only for metric/oracle selection and coverage auditing.
- Candidate construction uses DSE, BM25 and BGE-M3 scores only.
- The run stops on GPUs with less than 24 GB VRAM.
- Results are exploratory and must not be reported as a full ViDoRe V3 result.

Run every cell in order. The only interactive step is authorizing Google Drive.


In [ ]:
%pip install -q \
  "vidore-benchmark[all-retrievers]==5.0.0" \
  "colpali-engine==0.3.12" \
  "transformers==4.53.3" \
  "datasets>=2.19,<4" \
  "pyarrow>=15" \
  "PyYAML>=6"


In [ ]:
import gc
import hashlib
import importlib.metadata
import json
import os
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import yaml
from google.colab import drive

SEED = 20260820
DATASET = "vidore/vidore_v3_finance_en"
DATASET_REVISION = "7f432c176d82e27546501ad8064a713ac3071809"
MODEL_NAMES = {
    "bge_m3": "BAAI/bge-m3",
    "dse": "MrLight/dse-qwen2-2b-mrl-v1",
    "colqwen25": "vidore/colqwen2.5-v0.2",
}

if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU with at least 24 GB VRAM is required")
gpu = torch.cuda.get_device_properties(0)
gpu_gb = gpu.total_memory / 1024**3
if gpu_gb < 23.5:
    raise RuntimeError(f"GPU preflight failed: {gpu.name} has only {gpu_gb:.1f} GB VRAM")

drive.mount("/content/drive")
RUN_ROOT = Path("/content/drive/MyDrive/vidore_v3_finance_en_short_oracle")
CACHE_DIR = RUN_ROOT / "cache"
OUTPUT_DIR = RUN_ROOT / "output"
HF_HOME = RUN_ROOT / "hf_cache"
HF_HUB_CACHE = HF_HOME / "hub"
HF_DATASETS_CACHE = HF_HOME / "datasets"
for directory in [CACHE_DIR, OUTPUT_DIR, HF_HOME, HF_HUB_CACHE, HF_DATASETS_CACHE]:
    directory.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(HF_HOME)
os.environ["HF_HUB_CACHE"] = str(HF_HUB_CACHE)
os.environ["HF_DATASETS_CACHE"] = str(HF_DATASETS_CACHE)

# Import after cache variables are frozen so model and dataset downloads persist on Drive.
from huggingface_hub import model_info

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
print(f"GPU preflight passed: {gpu.name}, {gpu_gb:.1f} GB")
print(f"Persistent run directory: {RUN_ROOT}")


In [ ]:
# Install the exact local oracle implementation into this Colab runtime.
import sys
from pathlib import Path

package_root = Path("/content/oracle_study")
package_root.mkdir(parents=True, exist_ok=True)
embedded_files = {"__init__.py": "\"\"\"Oracle study tooling for visually rich document retrieval.\"\"\"\n\nfrom .constants import SEED\n\n__all__ = [\"SEED\"]\n\n", "bootstrap.py": "from __future__ import annotations\n\nimport numpy as np\n\nfrom .constants import SEED\n\n\ndef bootstrap_mean_ci(\n    values: np.ndarray,\n    n_resamples: int = 10_000,\n    seed: int = SEED,\n    confidence: float = 0.95,\n) -> tuple[float, float]:\n    values = np.asarray(values, dtype=float)\n    if values.size == 0:\n        raise ValueError(\"Cannot bootstrap an empty sample\")\n    rng = np.random.default_rng(seed)\n    estimates = np.empty(n_resamples, dtype=float)\n    batch = 256\n    for start in range(0, n_resamples, batch):\n        size = min(batch, n_resamples - start)\n        indices = rng.integers(0, values.size, size=(size, values.size))\n        estimates[start : start + size] = values[indices].mean(axis=1)\n    tail = (1.0 - confidence) / 2.0\n    return tuple(float(x) for x in np.quantile(estimates, [tail, 1.0 - tail]))\n\n\ndef bootstrap_ratio_ci(\n    numerator: np.ndarray,\n    denominator: np.ndarray,\n    n_resamples: int = 10_000,\n    seed: int = SEED,\n    confidence: float = 0.95,\n) -> tuple[float, float]:\n    numerator = np.asarray(numerator, dtype=float)\n    denominator = np.asarray(denominator, dtype=float)\n    if numerator.shape != denominator.shape or numerator.size == 0:\n        raise ValueError(\"Ratio bootstrap arrays must be non-empty and have equal shape\")\n    rng = np.random.default_rng(seed)\n    estimates = np.empty(n_resamples, dtype=float)\n    batch = 256\n    for start in range(0, n_resamples, batch):\n        size = min(batch, n_resamples - start)\n        indices = rng.integers(0, numerator.size, size=(size, numerator.size))\n        num = numerator[indices].mean(axis=1)\n        den = denominator[indices].mean(axis=1)\n        estimates[start : start + size] = np.divide(\n            num,\n            den,\n            out=np.ones_like(num),\n            where=np.abs(den) > 1e-15,\n        )\n    tail = (1.0 - confidence) / 2.0\n    return tuple(float(x) for x in np.quantile(estimates, [tail, 1.0 - tail]))\n\n", "constants.py": "from __future__ import annotations\n\nSEED = 20260820\n\nPREREGISTERED_THRESHOLDS = {\n    \"qpaf\": {\n        \"discovery_mean_delta_ndcg10_min\": 0.03,\n        \"confirmation_mean_delta_ndcg10_min\": 0.02,\n        \"confirmation_ci95_lower_min_exclusive\": 0.0,\n        \"confirmation_fraction_gain_ge_005_min\": 0.20,\n        \"no_go_confirmation_mean_delta_ndcg10_below\": 0.01,\n        \"no_go_top_5pct_gain_share_min\": 0.90,\n        \"must_hold_for_grids\": [\"w7\", \"w66\"],\n    },\n    \"budget_aware\": {\n        \"quality_absolute_tolerance\": 0.01,\n        \"beneficial_gain_exclusive\": 0.01,\n        \"full_minus_stage1_min\": 0.02,\n        \"beneficial_and_non_beneficial_fraction_min\": 0.20,\n        \"b_star_percent_max\": 50,\n        \"control_gap_at_b_star_min\": 0.01,\n        \"retention_ci95_lower_min\": 0.98,\n        \"no_go_full_minus_stage1_below\": 0.01,\n        \"no_go_b_star_percent_above\": 70,\n    },\n}\n\nSCORE_COLUMNS = [\n    \"dataset\",\n    \"query_id\",\n    \"page_id\",\n    \"relevance\",\n    \"bm25_score\",\n    \"dense_score\",\n    \"stage1_score\",\n    \"visual_score\",\n    \"branch_ranks\",\n]\n\nQUERY_METRIC_COLUMNS = [\n    \"dataset\",\n    \"query_id\",\n    \"source\",\n    \"relevant_count\",\n    \"ndcg_stage1\",\n    \"ndcg_full\",\n    \"recall_stage1\",\n    \"recall_full\",\n    \"stage1_margin\",\n    \"stage2_ms\",\n    \"stage2_flops\",\n]\n\nSECONDARY_QUERY_METRIC_COLUMNS = [\n    \"recall1_stage1\",\n    \"recall1_full\",\n    \"recall3_stage1\",\n    \"recall3_full\",\n    \"mrr10_stage1\",\n    \"mrr10_full\",\n]\n\nRAW_SCORE_COLUMNS = [\n    \"dataset\",\n    \"query_id\",\n    \"page_id\",\n    \"source\",\n    \"relevance\",\n    \"bm25_score\",\n    \"dense_score\",\n    \"stage1_score\",\n    \"visual_score\",\n    \"full_score\",\n]\n\nSCORE_NAMES = [\"bm25_score\", \"dense_score\", \"stage1_score\", \"visual_score\"]\n", "feasibility.py": "from __future__ import annotations\n\n\ndef _assess_direction(gain: dict, coverage: float) -> dict:\n    checks = {\n        \"candidate_coverage_ge_095\": coverage >= 0.95,\n        \"mean_gain_ge_001\": gain[\"mean_delta_ndcg10\"] >= 0.01,\n        \"ci95_lower_gt_zero\": gain[\"delta_ci95\"][0] > 0,\n        \"fraction_gain_ge_003_at_least_010\": gain[\"fraction_gain_ge_003\"] >= 0.10,\n        \"top_5pct_gain_share_below_090\": gain[\"top_5pct_gain_share\"] < 0.90,\n    }\n    feasible = all(checks.values())\n    strong = feasible and gain[\"mean_delta_ndcg10\"] >= 0.03\n    if not checks[\"candidate_coverage_ge_095\"]:\n        status = \"invalid_candidate_coverage\"\n    elif strong:\n        status = \"strong_signal_for_full_discovery\"\n    elif feasible:\n        status = \"feasible_signal\"\n    else:\n        status = \"no_clear_signal_on_finance_en_pilot\"\n    return {\"status\": status, \"checks\": checks, **gain}\n\n\ndef assess_short_pilot(oracle_summary: dict, coverage_report: dict, grid: str = \"w7\") -> dict:\n    if grid not in oracle_summary.get(\"grids\", {}):\n        raise ValueError(f\"Oracle summary does not contain grid {grid!r}\")\n    coverage = float(coverage_report[\"final_coverage\"])\n    grid_summary = oracle_summary[\"grids\"][grid]\n    return {\n        \"study\": \"vidore_v3_finance_en_short_oracle\",\n        \"scope\": \"exploratory_feasibility_not_formal_go_no_go\",\n        \"grid\": grid,\n        \"queries\": int(grid_summary[\"queries\"]),\n        \"candidate_coverage\": coverage,\n        \"qarf_vs_global\": _assess_direction(grid_summary[\"qarf_vs_global\"], coverage),\n        \"qpaf_vs_qarf\": _assess_direction(grid_summary[\"qpaf_vs_qarf\"], coverage),\n    }\n\n\ndef feasibility_markdown(report: dict) -> str:\n    lines = [\n        \"# ViDoRe V3 Finance EN — Short Oracle Feasibility\",\n        \"\",\n        \"> Exploratory pilot only. This report is not a formal Discovery GO/NO-GO decision.\",\n        \"\",\n        f\"- Queries: **{report['queries']}**\",\n        f\"- Candidate coverage: **{report['candidate_coverage']:.4f}**\",\n        f\"- Weight grid: **{report['grid']}**\",\n        \"\",\n        \"| Direction | Mean gain | 95% CI | Fraction gain >= .03 | Top-5% gain share | Status |\",\n        \"|---|---:|---:|---:|---:|---|\",\n    ]\n    for key, label in [(\"qarf_vs_global\", \"Global -> QARF\"), (\"qpaf_vs_qarf\", \"QARF -> QPAF\")]:\n        item = report[key]\n        lines.append(\n            f\"| {label} | {item['mean_delta_ndcg10']:.4f} | \"\n            f\"[{item['delta_ci95'][0]:.4f}, {item['delta_ci95'][1]:.4f}] | \"\n            f\"{item['fraction_gain_ge_003']:.3f} | {item['top_5pct_gain_share']:.3f} | \"\n            f\"`{item['status']}` |\"\n        )\n    lines.extend(\n        [\n            \"\",\n            \"A strong signal justifies the full ViDoRe V3 English Discovery study. A negative result here \"\n            \"must not be generalized beyond the finance_en corpus.\",\n            \"\",\n        ]\n    )\n    return \"\\n\".join(lines)\n", "io.py": "from __future__ import annotations\n\nimport hashlib\nimport json\nfrom pathlib import Path\nfrom typing import Iterable, Mapping\n\nimport pandas as pd\n\n\ndef read_table(path: str | Path) -> pd.DataFrame:\n    path = Path(path)\n    suffix = path.suffix.lower()\n    if suffix == \".parquet\":\n        return pd.read_parquet(path)\n    if suffix == \".csv\":\n        return pd.read_csv(path)\n    if suffix in {\".jsonl\", \".ndjson\"}:\n        return pd.read_json(path, lines=True)\n    raise ValueError(f\"Unsupported table format: {path}\")\n\n\ndef write_parquet(frame: pd.DataFrame, path: str | Path) -> None:\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    frame.to_parquet(path, index=False)\n\n\ndef write_json(data: Mapping | list, path: str | Path) -> None:\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temp = path.with_suffix(path.suffix + \".tmp\")\n    temp.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding=\"utf-8\")\n    temp.replace(path)\n\n\ndef write_jsonl(rows: Iterable[Mapping], path: str | Path) -> None:\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temp = path.with_suffix(path.suffix + \".tmp\")\n    with temp.open(\"w\", encoding=\"utf-8\", newline=\"\\n\") as handle:\n        for row in rows:\n            handle.write(json.dumps(row, ensure_ascii=False, separators=(\",\", \":\")) + \"\\n\")\n    temp.replace(path)\n\n\ndef require_columns(frame: pd.DataFrame, columns: Iterable[str], label: str) -> None:\n    missing = sorted(set(columns) - set(frame.columns))\n    if missing:\n        raise ValueError(f\"{label} is missing required columns: {missing}\")\n\n\ndef sha256_file(path: str | Path) -> str:\n    digest = hashlib.sha256()\n    with Path(path).open(\"rb\") as handle:\n        for chunk in iter(lambda: handle.read(1024 * 1024), b\"\"):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef dataframe_sha256(frame: pd.DataFrame, sort_by: list[str]) -> str:\n    \"\"\"Hash table content deterministically inside a frozen pandas environment.\"\"\"\n    ordered = frame.sort_values(sort_by, kind=\"stable\").reset_index(drop=True)\n    digest = hashlib.sha256()\n    digest.update(json.dumps(list(ordered.columns), separators=(\",\", \":\")).encode(\"utf-8\"))\n    digest.update(\n        json.dumps([str(dtype) for dtype in ordered.dtypes], separators=(\",\", \":\")).encode(\"utf-8\")\n    )\n    digest.update(pd.util.hash_pandas_object(ordered, index=False, categorize=True).values.tobytes())\n    return digest.hexdigest()\n", "metrics.py": "from __future__ import annotations\n\nimport math\nfrom dataclasses import dataclass\n\nimport numpy as np\n\n\n@dataclass(frozen=True)\nclass RankingMetrics:\n    ndcg10: float\n    recall1: float\n    recall3: float\n    mrr10: float\n\n\ndef stable_order(scores: np.ndarray, page_ids: np.ndarray) -> np.ndarray:\n    scores = np.asarray(scores, dtype=float)\n    page_ids = np.asarray(page_ids, dtype=str)\n    safe_scores = np.nan_to_num(scores, nan=-np.inf)\n    return np.lexsort((page_ids, -safe_scores))\n\n\ndef dcg_at_k(relevance: np.ndarray, k: int) -> float:\n    rel = np.asarray(relevance, dtype=float)[:k]\n    if rel.size == 0:\n        return 0.0\n    discounts = 1.0 / np.log2(np.arange(2, rel.size + 2, dtype=float))\n    return float(np.sum((np.power(2.0, rel) - 1.0) * discounts))\n\n\ndef ndcg_at_k(relevance: np.ndarray, k: int = 10) -> float:\n    rel = np.asarray(relevance, dtype=float)\n    ideal = dcg_at_k(np.sort(rel)[::-1], k)\n    if ideal <= 0:\n        return 0.0\n    return dcg_at_k(rel, k) / ideal\n\n\ndef recall_at_k(relevance: np.ndarray, k: int) -> float:\n    rel = np.asarray(relevance, dtype=float)\n    positives = int(np.count_nonzero(rel > 0))\n    if positives == 0:\n        return 0.0\n    return float(np.count_nonzero(rel[:k] > 0) / positives)\n\n\ndef mrr_at_k(relevance: np.ndarray, k: int = 10) -> float:\n    hits = np.flatnonzero(np.asarray(relevance, dtype=float)[:k] > 0)\n    return 0.0 if hits.size == 0 else 1.0 / float(hits[0] + 1)\n\n\ndef evaluate_scores(\n    scores: np.ndarray,\n    relevance: np.ndarray,\n    page_ids: np.ndarray,\n) -> RankingMetrics:\n    order = stable_order(scores, page_ids)\n    ranked_rel = np.asarray(relevance, dtype=float)[order]\n    return RankingMetrics(\n        ndcg10=ndcg_at_k(ranked_rel, 10),\n        recall1=recall_at_k(ranked_rel, 1),\n        recall3=recall_at_k(ranked_rel, 3),\n        mrr10=mrr_at_k(ranked_rel, 10),\n    )\n\n\ndef minmax(values: np.ndarray) -> np.ndarray:\n    values = np.asarray(values, dtype=float)\n    if values.size == 0:\n        return values.copy()\n    low = float(np.nanmin(values))\n    high = float(np.nanmax(values))\n    if not math.isfinite(low) or not math.isfinite(high):\n        raise ValueError(\"Scores must be finite before normalization\")\n    if math.isclose(low, high, rel_tol=0.0, abs_tol=1e-15):\n        return np.zeros_like(values, dtype=float)\n    return (values - low) / (high - low)\n\n", "profiles.py": "from __future__ import annotations\n\nfrom itertools import product\n\nimport numpy as np\n\n\nW7 = np.asarray(\n    [\n        (1.0, 0.0, 0.0),\n        (0.0, 1.0, 0.0),\n        (0.0, 0.0, 1.0),\n        (0.5, 0.5, 0.0),\n        (0.5, 0.0, 0.5),\n        (0.0, 0.5, 0.5),\n        (1.0 / 3.0, 1.0 / 3.0, 1.0 / 3.0),\n    ],\n    dtype=float,\n)\n\n\ndef w66() -> np.ndarray:\n    values = []\n    for a, b, c in product(range(11), repeat=3):\n        if a + b + c == 10:\n            values.append((a / 10.0, b / 10.0, c / 10.0))\n    return np.asarray(values, dtype=float)\n\n\ndef get_profiles(name: str) -> np.ndarray:\n    lowered = name.lower()\n    if lowered == \"w7\":\n        return W7.copy()\n    if lowered == \"w66\":\n        return w66()\n    raise ValueError(f\"Unknown profile grid: {name}\")\n\n", "qpaf.py": "from __future__ import annotations\n\nimport bisect\nfrom dataclasses import asdict\n\nimport numpy as np\nimport pandas as pd\n\nfrom .bootstrap import bootstrap_mean_ci\nfrom .constants import PREREGISTERED_THRESHOLDS, SCORE_COLUMNS\nfrom .io import require_columns\nfrom .metrics import RankingMetrics, dcg_at_k, evaluate_scores, stable_order\nfrom .profiles import get_profiles\n\n\nTOLERANCE = 1e-12\n\n\ndef _better_metrics(candidate: RankingMetrics, current: RankingMetrics) -> bool:\n    candidate_key = (candidate.ndcg10, candidate.recall3, candidate.mrr10)\n    current_key = (current.ndcg10, current.recall3, current.mrr10)\n    for left, right in zip(candidate_key, current_key):\n        if left > right + TOLERANCE:\n            return True\n        if left < right - TOLERANCE:\n            return False\n    return False\n\n\ndef _query_oracle(\n    score_matrix: np.ndarray,\n    relevance: np.ndarray,\n    page_ids: np.ndarray,\n    profiles: np.ndarray,\n) -> tuple[int, np.ndarray, RankingMetrics]:\n    best_index = 0\n    best_scores = score_matrix @ profiles[0]\n    best_metrics = evaluate_scores(best_scores, relevance, page_ids)\n    for index in range(1, len(profiles)):\n        scores = score_matrix @ profiles[index]\n        metrics = evaluate_scores(scores, relevance, page_ids)\n        if _better_metrics(metrics, best_metrics):\n            best_index = index\n            best_scores = scores\n            best_metrics = metrics\n    return best_index, best_scores, best_metrics\n\n\ndef _mean_metrics(metrics: list[RankingMetrics]) -> RankingMetrics:\n    return RankingMetrics(\n        ndcg10=float(np.mean([item.ndcg10 for item in metrics])),\n        recall1=float(np.mean([item.recall1 for item in metrics])),\n        recall3=float(np.mean([item.recall3 for item in metrics])),\n        mrr10=float(np.mean([item.mrr10 for item in metrics])),\n    )\n\n\ndef _global_oracle(\n    query_data: list[dict],\n    profiles: np.ndarray,\n) -> tuple[int, list[RankingMetrics]]:\n    \"\"\"Choose one profile for every query, with the same metric tie-break as QARF.\"\"\"\n    best_index = 0\n    best_per_query = [\n        evaluate_scores(item[\"score_matrix\"] @ profiles[0], item[\"relevance\"], item[\"page_ids\"])\n        for item in query_data\n    ]\n    best_mean = _mean_metrics(best_per_query)\n    for index in range(1, len(profiles)):\n        per_query = [\n            evaluate_scores(item[\"score_matrix\"] @ profiles[index], item[\"relevance\"], item[\"page_ids\"])\n            for item in query_data\n        ]\n        mean = _mean_metrics(per_query)\n        if _better_metrics(mean, best_mean):\n            best_index = index\n            best_per_query = per_query\n            best_mean = mean\n    return best_index, best_per_query\n\n\ndef _discounts(length: int, k: int = 10) -> np.ndarray:\n    discounts = np.zeros(length + 1, dtype=float)\n    active = min(length, k)\n    discounts[:active] = 1.0 / np.log2(np.arange(2, active + 2, dtype=float))\n    return discounts\n\n\ndef _single_item_ndcg(\n    current_scores: np.ndarray,\n    relevance: np.ndarray,\n    page_ids: np.ndarray,\n    candidate_index: int,\n    new_score: float,\n    current_ndcg: float,\n    k: int = 10,\n) -> float:\n    order = stable_order(current_scores, page_ids)\n    positions = np.empty(len(order), dtype=int)\n    positions[order] = np.arange(len(order))\n    old_position = int(positions[candidate_index])\n    other_order = order[order != candidate_index]\n    other_keys = [(-float(current_scores[i]), str(page_ids[i])) for i in other_order]\n    new_position = bisect.bisect_left(other_keys, (-float(new_score), str(page_ids[candidate_index])))\n    if new_position == old_position:\n        return current_ndcg\n\n    gains = np.power(2.0, relevance.astype(float)) - 1.0\n    ranked_gains = gains[order]\n    discounts = _discounts(len(order), k)\n    ideal_dcg = dcg_at_k(np.sort(relevance)[::-1], k)\n    if ideal_dcg <= 0:\n        return 0.0\n    current_dcg = current_ndcg * ideal_dcg\n    item_gain = gains[candidate_index]\n    delta = item_gain * (discounts[new_position] - discounts[old_position])\n\n    if new_position < old_position:\n        shifts = ranked_gains * (discounts[1:] - discounts[:-1])\n        delta += float(shifts[new_position:old_position].sum())\n    else:\n        shifts = np.zeros(len(order), dtype=float)\n        shifts[1:] = ranked_gains[1:] * (discounts[:-2] - discounts[1:-1])\n        delta += float(shifts[old_position + 1 : new_position + 1].sum())\n    return max(0.0, min(1.0, (current_dcg + delta) / ideal_dcg))\n\n\ndef _candidate_oracle(\n    score_matrix: np.ndarray,\n    relevance: np.ndarray,\n    page_ids: np.ndarray,\n    profiles: np.ndarray,\n    query_profile_index: int,\n    query_scores: np.ndarray,\n    query_metrics: RankingMetrics,\n    max_sweeps: int = 2,\n) -> tuple[np.ndarray, np.ndarray, RankingMetrics, int]:\n    assignments = np.full(len(page_ids), query_profile_index, dtype=int)\n    current_scores = query_scores.copy()\n    current_metrics = query_metrics\n    fixed_order = stable_order(query_scores, page_ids)\n    changes = 0\n\n    for _ in range(max_sweeps):\n        changed_this_sweep = 0\n        for candidate_index in fixed_order:\n            best_profile = int(assignments[candidate_index])\n            best_ndcg = current_metrics.ndcg10\n            best_score = float(current_scores[candidate_index])\n            for profile_index, profile in enumerate(profiles):\n                if profile_index == assignments[candidate_index]:\n                    continue\n                new_score = float(score_matrix[candidate_index] @ profile)\n                ndcg = _single_item_ndcg(\n                    current_scores,\n                    relevance,\n                    page_ids,\n                    int(candidate_index),\n                    new_score,\n                    current_metrics.ndcg10,\n                )\n                if ndcg > best_ndcg + TOLERANCE:\n                    best_profile = profile_index\n                    best_ndcg = ndcg\n                    best_score = new_score\n            if best_profile != assignments[candidate_index]:\n                assignments[candidate_index] = best_profile\n                current_scores[candidate_index] = best_score\n                current_metrics = evaluate_scores(current_scores, relevance, page_ids)\n                changed_this_sweep += 1\n                changes += 1\n        if changed_this_sweep == 0:\n            break\n    return assignments, current_scores, current_metrics, changes\n\n\ndef run_qpaf_oracle(\n    scores: pd.DataFrame,\n    grids: tuple[str, ...] = (\"w7\", \"w66\"),\n    n_bootstrap: int = 10_000,\n) -> tuple[list[dict], dict, pd.DataFrame]:\n    require_columns(scores, SCORE_COLUMNS, \"retrieval score table\")\n    if scores.empty:\n        raise ValueError(\"QPAF oracle requires at least one candidate row\")\n    query_data: list[dict] = []\n    for (dataset, query_id), group in scores.groupby([\"dataset\", \"query_id\"], sort=False):\n        group = group.sort_values(\"page_id\", kind=\"stable\")\n        query_data.append(\n            {\n                \"dataset\": str(dataset),\n                \"query_id\": str(query_id),\n                \"source\": str(group[\"source\"].iloc[0]) if \"source\" in group else \"unknown\",\n                \"score_matrix\": group[[\"bm25_score\", \"dense_score\", \"visual_score\"]].to_numpy(float),\n                \"relevance\": group[\"relevance\"].to_numpy(float),\n                \"page_ids\": group[\"page_id\"].astype(str).to_numpy(),\n            }\n        )\n\n    rows: list[dict] = []\n    for grid_name in grids:\n        profiles = get_profiles(grid_name)\n        global_profile, global_metrics = _global_oracle(query_data, profiles)\n        for item, global_query_metrics in zip(query_data, global_metrics):\n            score_matrix = item[\"score_matrix\"]\n            relevance = item[\"relevance\"]\n            page_ids = item[\"page_ids\"]\n            query_profile, query_scores, query_metrics = _query_oracle(\n                score_matrix, relevance, page_ids, profiles\n            )\n            assignments, _, candidate_metrics, changes = _candidate_oracle(\n                score_matrix,\n                relevance,\n                page_ids,\n                profiles,\n                query_profile,\n                query_scores,\n                query_metrics,\n            )\n            changed_mask = assignments != query_profile\n            changed_pages = {\n                str(page_ids[index]): [float(x) for x in profiles[assignments[index]]]\n                for index in np.flatnonzero(changed_mask)\n            }\n            counts = np.bincount(assignments, minlength=len(profiles))\n            rows.append(\n                {\n                    \"dataset\": item[\"dataset\"],\n                    \"query_id\": item[\"query_id\"],\n                    \"source\": item[\"source\"],\n                    \"relevant_count\": int(np.count_nonzero(relevance > 0)),\n                    \"grid\": grid_name,\n                    \"global_profile\": [float(x) for x in profiles[global_profile]],\n                    \"global_metrics\": asdict(global_query_metrics),\n                    \"qarf_profile\": [float(x) for x in profiles[query_profile]],\n                    \"qarf_metrics\": asdict(query_metrics),\n                    \"qpaf_metrics\": asdict(candidate_metrics),\n                    \"delta_qarf_vs_global\": query_metrics.ndcg10 - global_query_metrics.ndcg10,\n                    \"delta_qpaf_vs_qarf\": candidate_metrics.ndcg10 - query_metrics.ndcg10,\n                    \"delta_qpaf_vs_global\": candidate_metrics.ndcg10 - global_query_metrics.ndcg10,\n                    # Backward-compatible alias used by the original QPAF reports.\n                    \"delta_ndcg10\": candidate_metrics.ndcg10 - query_metrics.ndcg10,\n                    \"changed_candidates\": int(np.count_nonzero(changed_mask)),\n                    \"accepted_updates\": int(changes),\n                    \"profile_counts\": counts.astype(int).tolist(),\n                    \"changed_pages\": changed_pages,\n                }\n            )\n\n    result_frame = pd.DataFrame(\n        [\n            {\n                \"dataset\": row[\"dataset\"],\n                \"query_id\": row[\"query_id\"],\n                \"source\": row[\"source\"],\n                \"relevant_count\": row[\"relevant_count\"],\n                \"grid\": row[\"grid\"],\n                \"global_ndcg10\": row[\"global_metrics\"][\"ndcg10\"],\n                \"qarf_ndcg10\": row[\"qarf_metrics\"][\"ndcg10\"],\n                \"qpaf_ndcg10\": row[\"qpaf_metrics\"][\"ndcg10\"],\n                \"delta_qarf_vs_global\": row[\"delta_qarf_vs_global\"],\n                \"delta_qpaf_vs_qarf\": row[\"delta_qpaf_vs_qarf\"],\n                \"delta_qpaf_vs_global\": row[\"delta_qpaf_vs_global\"],\n                \"delta_ndcg10\": row[\"delta_ndcg10\"],\n                \"changed_candidates\": row[\"changed_candidates\"],\n            }\n            for row in rows\n        ]\n    )\n    summaries = {}\n    for grid_name, group in result_frame.groupby(\"grid\", sort=False):\n        qarf_delta = group[\"delta_qarf_vs_global\"].to_numpy(float)\n        qpaf_delta = group[\"delta_qpaf_vs_qarf\"].to_numpy(float)\n\n        def gain_summary(delta: np.ndarray) -> dict:\n            ci_low, ci_high = bootstrap_mean_ci(delta, n_bootstrap)\n            positive = np.clip(delta, 0.0, None)\n            top_count = max(1, int(np.ceil(len(positive) * 0.05)))\n            positive_total = float(positive.sum())\n            concentration = (\n                float(np.sort(positive)[-top_count:].sum() / positive_total)\n                if positive_total > 0\n                else 0.0\n            )\n            return {\n                \"mean_delta_ndcg10\": float(delta.mean()),\n                \"delta_ci95\": [ci_low, ci_high],\n                \"fraction_gain_ge_001\": float(np.mean(delta >= 0.01)),\n                \"fraction_gain_ge_003\": float(np.mean(delta >= 0.03)),\n                \"fraction_gain_ge_005\": float(np.mean(delta >= 0.05)),\n                \"top_5pct_gain_share\": concentration,\n            }\n\n        qarf_gain = gain_summary(qarf_delta)\n        qpaf_gain = gain_summary(qpaf_delta)\n        grid_rows = [row for row in rows if row[\"grid\"] == grid_name]\n        profile_totals = np.sum(\n            np.asarray([row[\"profile_counts\"] for row in grid_rows], dtype=int),\n            axis=0,\n        )\n        summaries[grid_name] = {\n            \"queries\": int(len(group)),\n            \"global_profile\": grid_rows[0][\"global_profile\"],\n            \"mean_global_ndcg10\": float(group[\"global_ndcg10\"].mean()),\n            \"mean_qarf_ndcg10\": float(group[\"qarf_ndcg10\"].mean()),\n            \"mean_qpaf_ndcg10\": float(group[\"qpaf_ndcg10\"].mean()),\n            \"qarf_vs_global\": qarf_gain,\n            \"qpaf_vs_qarf\": qpaf_gain,\n            \"qpaf_vs_global\": gain_summary(group[\"delta_qpaf_vs_global\"].to_numpy(float)),\n            # Backward-compatible QPAF-vs-QARF summary fields.\n            **qpaf_gain,\n            \"mean_changed_candidates\": float(group[\"changed_candidates\"].mean()),\n            \"profile_counts\": profile_totals.astype(int).tolist(),\n        }\n    primary = summaries.get(\"w7\", next(iter(summaries.values())))\n    discovery_min = PREREGISTERED_THRESHOLDS[\"qpaf\"][\"discovery_mean_delta_ndcg10_min\"]\n    required_sensitivity = [grid for grid in [\"w7\", \"w66\"] if grid in summaries]\n    discovery_holds_across_grids = len(required_sensitivity) == 2 and all(\n        summaries[grid][\"mean_delta_ndcg10\"] >= discovery_min\n        and summaries[grid][\"delta_ci95\"][0] > 0\n        for grid in required_sensitivity\n    )\n    if discovery_holds_across_grids:\n        discovery_gate = \"pass\"\n    elif (\n        primary[\"mean_delta_ndcg10\"]\n        < PREREGISTERED_THRESHOLDS[\"qpaf\"][\"no_go_confirmation_mean_delta_ndcg10_below\"]\n    ):\n        discovery_gate = \"fail\"\n    else:\n        discovery_gate = \"inconclusive\"\n    summary = {\"grids\": summaries, \"discovery_gate\": discovery_gate}\n\n    subgroup = (\n        result_frame.groupby([\"grid\", \"source\", result_frame[\"relevant_count\"].gt(1).map({True: \"multi\", False: \"single\"})])\n        .agg(\n            queries=(\"query_id\", \"size\"),\n            mean_delta_qarf_vs_global=(\"delta_qarf_vs_global\", \"mean\"),\n            mean_delta_qpaf_vs_qarf=(\"delta_qpaf_vs_qarf\", \"mean\"),\n            mean_delta_qpaf_vs_global=(\"delta_qpaf_vs_global\", \"mean\"),\n        )\n        .reset_index(names=[\"grid\", \"source\", \"relevance_group\"])\n    )\n    return rows, summary, subgroup\n", "report.py": "from __future__ import annotations\n\nimport json\nfrom pathlib import Path\n\nimport matplotlib\n\nmatplotlib.use(\"Agg\")\nimport matplotlib.pyplot as plt\nimport pandas as pd\n\n\ndef plot_qpaf(result_frame: pd.DataFrame, output: str | Path) -> None:\n    output = Path(output)\n    output.parent.mkdir(parents=True, exist_ok=True)\n    fig, axes = plt.subplots(1, 3, figsize=(14, 4))\n    for grid, group in result_frame.groupby(\"grid\", sort=False):\n        axes[0].hist(group[\"delta_qarf_vs_global\"], bins=20, alpha=0.55, label=f\"QARF {grid}\")\n        axes[0].hist(group[\"delta_qpaf_vs_qarf\"], bins=20, alpha=0.45, label=f\"QPAF {grid}\")\n        axes[1].scatter(group[\"global_ndcg10\"], group[\"qarf_ndcg10\"], s=9, alpha=0.45, label=grid)\n        axes[2].scatter(group[\"qarf_ndcg10\"], group[\"qpaf_ndcg10\"], s=9, alpha=0.45, label=grid)\n    axes[0].set_title(\"Adaptive fusion oracle gain\")\n    axes[0].set_xlabel(\"Δ nDCG@10\")\n    axes[0].set_ylabel(\"Queries\")\n    axes[1].plot([0, 1], [0, 1], color=\"black\", linewidth=1)\n    axes[1].set_title(\"Global vs QARF\")\n    axes[1].set_xlabel(\"Global nDCG@10\")\n    axes[1].set_ylabel(\"QARF nDCG@10\")\n    axes[2].plot([0, 1], [0, 1], color=\"black\", linewidth=1)\n    axes[2].set_title(\"QARF vs QPAF\")\n    axes[2].set_xlabel(\"QARF nDCG@10\")\n    axes[2].set_ylabel(\"QPAF nDCG@10\")\n    for axis in axes:\n        axis.legend()\n        axis.grid(alpha=0.2)\n    fig.tight_layout()\n    fig.savefig(output, dpi=160)\n    plt.close(fig)\n\n\ndef plot_budget(curve: pd.DataFrame, output: str | Path) -> None:\n    output = Path(output)\n    output.parent.mkdir(parents=True, exist_ok=True)\n    fig, axis = plt.subplots(figsize=(7, 4.5))\n    axis.plot(curve[\"budget_percent\"], curve[\"oracle_ndcg10\"], marker=\"o\", label=\"Oracle\")\n    axis.plot(curve[\"budget_percent\"], curve[\"margin_ndcg10\"], label=\"Stage-1 margin\")\n    axis.plot(curve[\"budget_percent\"], curve[\"random_ndcg10\"], label=\"Random\")\n    axis.axhline(curve[\"full_ndcg10\"].iloc[0], color=\"black\", linestyle=\"--\", label=\"Full HEAVEN\")\n    axis.axhline(curve[\"cheap_ndcg10\"].iloc[0], color=\"gray\", linestyle=\":\", label=\"Stage 1\")\n    axis.set_xlabel(\"Stage-2 invocation budget (%)\")\n    axis.set_ylabel(\"Mean nDCG@10\")\n    axis.set_title(\"Budget-aware oracle curve\")\n    axis.grid(alpha=0.2)\n    axis.legend()\n    fig.tight_layout()\n    fig.savefig(output, dpi=160)\n    plt.close(fig)\n\n\ndef plot_budget_cost(curve: pd.DataFrame, output: str | Path) -> None:\n    output = Path(output)\n    output.parent.mkdir(parents=True, exist_ok=True)\n    fig, axes = plt.subplots(1, 2, figsize=(10, 4))\n    axes[0].plot(\n        curve[\"latency_efficient_stage2_ms\"],\n        curve[\"latency_efficient_ndcg10\"],\n        marker=\"o\",\n    )\n    axes[0].set_xlabel(\"Cumulative Stage-2 latency (ms)\")\n    axes[0].set_ylabel(\"Mean nDCG@10\")\n    axes[0].set_title(\"Gain / latency oracle\")\n    axes[1].plot(\n        curve[\"flops_efficient_stage2_flops\"],\n        curve[\"flops_efficient_ndcg10\"],\n        marker=\"o\",\n    )\n    axes[1].set_xlabel(\"Cumulative Stage-2 FLOPs\")\n    axes[1].set_ylabel(\"Mean nDCG@10\")\n    axes[1].set_title(\"Gain / FLOPs oracle\")\n    for axis in axes:\n        axis.grid(alpha=0.2)\n    fig.tight_layout()\n    fig.savefig(output, dpi=160)\n    plt.close(fig)\n\n\ndef decision_markdown(decision: dict) -> str:\n    return (\n        \"# Oracle Study Decision\\n\\n\"\n        f\"- QPAF gate: **{decision['qpaf_gate']}**\\n\"\n        f\"- Budget-Aware gate: **{decision['budget_gate']}**\\n\"\n        f\"- Recommended direction: **{decision['recommended_direction']}**\\n\\n\"\n        \"The decision uses the preregistered thresholds. Do not reinterpret a failed or \"\n        \"inconclusive gate as a positive result.\\n\\n\"\n        \"## Preregistered thresholds\\n\\n\"\n        \"```json\\n\"\n        f\"{json.dumps(decision['preregistered_thresholds'], indent=2)}\\n\"\n        \"```\\n\\n\"\n        \"## Observed summaries\\n\\n\"\n        \"```json\\n\"\n        f\"{json.dumps(decision['observed'], indent=2)}\\n\"\n        \"```\\n\"\n    )\n"}
for filename, content in embedded_files.items():
    (package_root / filename).write_text(content, encoding="utf-8")
if "/content" not in sys.path:
    sys.path.insert(0, "/content")

from oracle_study.feasibility import assess_short_pilot, feasibility_markdown
from oracle_study.io import dataframe_sha256, write_json, write_jsonl
from oracle_study.qpaf import run_qpaf_oracle
from oracle_study.report import plot_qpaf


In [ ]:
from datasets import load_dataset

corpus = load_dataset(DATASET, "corpus", split="test", revision=DATASET_REVISION)
query_dataset = load_dataset(DATASET, "queries", split="test", revision=DATASET_REVISION)
qrel_dataset = load_dataset(DATASET, "qrels", split="test", revision=DATASET_REVISION)

query_frame = query_dataset.to_pandas()
query_frame = query_frame.loc[query_frame["language"].eq("english")].copy()
query_frame = query_frame.sort_values("query_id", kind="stable").reset_index(drop=True)
qrel_frame = qrel_dataset.to_pandas()
qrel_frame = qrel_frame.loc[
    qrel_frame["query_id"].isin(query_frame["query_id"]) & qrel_frame["score"].gt(0)
].copy()

corpus_ids = np.asarray(corpus["corpus_id"])
query_ids = query_frame["query_id"].to_numpy()
query_texts = query_frame["query"].astype(str).tolist()
corpus_texts = ["" if value is None else str(value) for value in corpus["markdown"]]

assert len(corpus_ids) == 2942, len(corpus_ids)
assert len(query_ids) == 309, len(query_ids)
assert len(set(corpus_ids.tolist())) == len(corpus_ids)
assert len(set(query_ids.tolist())) == len(query_ids)
assert set(qrel_frame["query_id"]).issubset(set(query_ids))
assert set(qrel_frame["corpus_id"]).issubset(set(corpus_ids))
assert qrel_frame.groupby("query_id")["corpus_id"].nunique().reindex(query_ids).notna().all()

qrel_lookup = {
    (row.query_id, row.corpus_id): float(row.score)
    for row in qrel_frame.itertuples(index=False)
}
relevant_count = qrel_frame.groupby("query_id")["corpus_id"].nunique().to_dict()

def source_label(value):
    if isinstance(value, np.ndarray):
        values = value.tolist()
    elif isinstance(value, (list, tuple)):
        values = list(value)
    else:
        values = [value]
    return "|".join(sorted({str(item) for item in values if item is not None})) or "unknown"

query_sources = [source_label(value) for value in query_frame["content_type"]]
print(f"Mapping smoke test passed: {len(query_ids)} English queries, {len(corpus_ids)} pages")
print(f"Positive qrels: {len(qrel_frame)}; mean relevant pages/query: {np.mean(list(relevant_count.values())):.2f}")


In [ ]:
BATCH_SIZES = {}

def load_torch(path):
    return torch.load(path, map_location="cpu", weights_only=False)

def encode_with_backoff(label, function, values, cache_path, batch_candidates):
    cache_path = Path(cache_path)
    if cache_path.exists():
        BATCH_SIZES[label] = "cache"
        return load_torch(cache_path)
    last_error = None
    for batch_size in batch_candidates:
        try:
            output = function(values, batch_size=batch_size)
            torch.save(output, cache_path)
            BATCH_SIZES[label] = batch_size
            return output
        except (torch.cuda.OutOfMemoryError, RuntimeError) as error:
            if "out of memory" not in str(error).lower():
                raise
            last_error = error
            gc.collect()
            torch.cuda.empty_cache()
            print(f"{label}: OOM at batch {batch_size}; retrying smaller batch")
    raise RuntimeError(f"{label}: all approved batch sizes failed") from last_error

def checkpoint_sha(model_name):
    return model_info(model_name).sha

def stable_top_indices(scores, k):
    page_keys = corpus_ids.astype(str)
    return np.lexsort((page_keys, -np.asarray(scores, dtype=float)))[:k]

def minmax(values):
    values = np.asarray(values, dtype=float)
    low, high = float(values.min()), float(values.max())
    return np.zeros_like(values) if abs(high - low) <= 1e-15 else (values - low) / (high - low)

class LazyCorpusImages:
    # Decode only the page images requested by the current model batch.

    def __init__(self, dataset):
        self.dataset = dataset

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, index):
        if isinstance(index, slice):
            return [self[position] for position in range(*index.indices(len(self)))]
        return self.dataset[int(index)]["image"].convert("RGB")

    def __iter__(self):
        for index in range(len(self)):
            yield self[index]

# The first ten queries form the fixed model smoke set.
smoke_queries = query_texts[:10]


## 1. BM25 over the provided OCR markdown

In [ ]:
import nltk
from vidore_benchmark.retrievers.bm25_retriever import BM25Retriever

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)
bm25_path = CACHE_DIR / "bm25_scores.pt"
if bm25_path.exists():
    bm25_scores = load_torch(bm25_path)
else:
    retriever = BM25Retriever(device="cpu")
    smoke = retriever.get_scores_bm25(smoke_queries, corpus_texts[:32])
    assert smoke.shape == (10, 32)
    bm25_scores = retriever.get_scores_bm25(query_texts, corpus_texts)
    torch.save(bm25_scores, bm25_path)
    del retriever, smoke
bm25_scores = bm25_scores.float().numpy()
assert bm25_scores.shape == (309, 2942)
print("BM25 scores ready", bm25_scores.shape)


## 2. BGE-M3 dense OCR retrieval

In [ ]:
from vidore_benchmark.retrievers.bge_m3_retriever import BGEM3Retriever

bge_score_path = CACHE_DIR / "bge_m3_scores.pt"
if bge_score_path.exists():
    dense_scores = load_torch(bge_score_path)
else:
    retriever = BGEM3Retriever(pretrained_model_name_or_path=MODEL_NAMES["bge_m3"], device="cuda")
    smoke_q = retriever.forward_queries(smoke_queries, batch_size=8)
    smoke_p = retriever.forward_passages(corpus_texts[:32], batch_size=8)
    assert retriever.get_scores(smoke_q, smoke_p).shape == (10, 32)
    passage_embeddings = encode_with_backoff(
        "bge_passages", retriever.forward_passages, corpus_texts,
        CACHE_DIR / "bge_m3_passage_embeddings.pt", [32, 16, 8]
    )
    query_embeddings = encode_with_backoff(
        "bge_queries", retriever.forward_queries, query_texts,
        CACHE_DIR / "bge_m3_query_embeddings.pt", [64, 32, 16]
    )
    dense_scores = retriever.get_scores(query_embeddings, passage_embeddings)
    torch.save(dense_scores, bge_score_path)
    del retriever, smoke_q, smoke_p, passage_embeddings, query_embeddings
    gc.collect()
    torch.cuda.empty_cache()
dense_scores = dense_scores.float().numpy()
assert dense_scores.shape == (309, 2942)
print("BGE-M3 scores ready", dense_scores.shape)


## 3. DSE single-vector visual retrieval and candidate construction

In [ ]:
from vidore_benchmark.retrievers.dse_qwen2_retriever import DSEQwen2Retriever

dse_score_path = CACHE_DIR / "dse_scores.pt"
if dse_score_path.exists():
    stage1_scores = load_torch(dse_score_path)
else:
    retriever = DSEQwen2Retriever(
        pretrained_model_name_or_path=MODEL_NAMES["dse"], num_image_tokens=1024, device="cuda"
    )
    smoke_images = [corpus[index]["image"].convert("RGB") for index in range(8)]
    smoke_q = retriever.forward_queries(smoke_queries, batch_size=4)
    smoke_p = retriever.forward_passages(smoke_images, batch_size=2)
    assert retriever.get_scores(smoke_q, smoke_p).shape == (10, 8)
    corpus_images = LazyCorpusImages(corpus)
    passage_embeddings = encode_with_backoff(
        "dse_passages", retriever.forward_passages, corpus_images,
        CACHE_DIR / "dse_passage_embeddings.pt", [4, 2, 1]
    )
    query_embeddings = encode_with_backoff(
        "dse_queries", retriever.forward_queries, query_texts,
        CACHE_DIR / "dse_query_embeddings.pt", [16, 8, 4]
    )
    stage1_scores = retriever.get_scores(query_embeddings, passage_embeddings)
    torch.save(stage1_scores, dse_score_path)
    del (
        retriever, smoke_q, smoke_p, smoke_images, corpus_images,
        passage_embeddings, query_embeddings
    )
    gc.collect()
    torch.cuda.empty_cache()
stage1_scores = stage1_scores.float().numpy()
assert stage1_scores.shape == (309, 2942)

def make_candidate_indices(stage1_k, bm25_k, dense_k):
    return [
        np.unique(np.concatenate([
            stable_top_indices(stage1_scores[index], stage1_k),
            stable_top_indices(bm25_scores[index], bm25_k),
            stable_top_indices(dense_scores[index], dense_k),
        ]))
        for index in range(len(query_ids))
    ]

def candidate_coverage(candidate_indices):
    selected = {
        (query_ids[index], corpus_ids[page_index])
        for index, pages in enumerate(candidate_indices)
        for page_index in pages
    }
    relevant = set(qrel_lookup)
    return len(selected & relevant) / len(relevant)

initial_candidates = make_candidate_indices(200, 100, 100)
initial_coverage = candidate_coverage(initial_candidates)
expanded_once = initial_coverage < 0.95
candidate_indices = make_candidate_indices(300, 200, 200) if expanded_once else initial_candidates
final_coverage = candidate_coverage(candidate_indices)
coverage_report = {
    "dataset": DATASET,
    "minimum_required": 0.95,
    "initial_coverage": initial_coverage,
    "final_coverage": final_coverage,
    "expanded_once": expanded_once,
    "queries": len(query_ids),
    "corpus_pages": len(corpus_ids),
    "mean_candidates": float(np.mean([len(value) for value in candidate_indices])),
}
write_json(coverage_report, OUTPUT_DIR / "coverage_report.json")
if final_coverage < 0.95:
    raise RuntimeError(f"Candidate coverage gate failed: {final_coverage:.4f}")
print(coverage_report)


## 4. ColQwen2.5 multi-vector candidate scoring

In [ ]:
from colpali_engine.models import ColQwen2_5_Processor
from vidore_benchmark.retrievers.colqwen2_5_retriever import ColQwen2_5_Retriever

colqwen_passage_path = CACHE_DIR / "colqwen25_passage_embeddings.pt"
colqwen_query_path = CACHE_DIR / "colqwen25_query_embeddings.pt"
if not colqwen_passage_path.exists() or not colqwen_query_path.exists():
    retriever = ColQwen2_5_Retriever(
        pretrained_model_name_or_path=MODEL_NAMES["colqwen25"], device="cuda", num_workers=0
    )
    smoke_images = [corpus[index]["image"].convert("RGB") for index in range(4)]
    smoke_q = retriever.forward_queries(smoke_queries, batch_size=2)
    smoke_p = retriever.forward_passages(smoke_images, batch_size=1)
    assert retriever.get_scores(smoke_q, smoke_p, batch_size=8).shape == (10, 4)
    corpus_images = LazyCorpusImages(corpus)
    passage_embeddings = encode_with_backoff(
        "colqwen25_passages", retriever.forward_passages, corpus_images,
        colqwen_passage_path, [2, 1]
    )
    query_embeddings = encode_with_backoff(
        "colqwen25_queries", retriever.forward_queries, query_texts,
        colqwen_query_path, [8, 4, 2]
    )
    del retriever, smoke_q, smoke_p, smoke_images, corpus_images
    gc.collect()
    torch.cuda.empty_cache()
else:
    passage_embeddings = load_torch(colqwen_passage_path)
    query_embeddings = load_torch(colqwen_query_path)

processor = ColQwen2_5_Processor.from_pretrained(MODEL_NAMES["colqwen25"])
visual_score_path = CACHE_DIR / "colqwen25_candidate_scores.pt"
if visual_score_path.exists():
    visual_scores = load_torch(visual_score_path).numpy()
else:
    visual_scores = np.full((len(query_ids), len(corpus_ids)), np.nan, dtype=np.float32)

for query_index, pages in enumerate(candidate_indices):
    if np.isfinite(visual_scores[query_index, pages]).all():
        continue
    selected_embeddings = [passage_embeddings[int(index)] for index in pages]
    score = processor.score(
        [query_embeddings[query_index]], selected_embeddings, batch_size=128, device="cpu"
    )[0].float().numpy()
    visual_scores[query_index, pages] = score
    if (query_index + 1) % 10 == 0:
        torch.save(torch.from_numpy(visual_scores), visual_score_path)
        print(f"ColQwen candidate scores: {query_index + 1}/{len(query_ids)}")
torch.save(torch.from_numpy(visual_scores), visual_score_path)
assert all(np.isfinite(visual_scores[index, pages]).all() for index, pages in enumerate(candidate_indices))
del processor, passage_embeddings, query_embeddings
gc.collect()
print("ColQwen2.5 candidate scores ready")


## 5. Build the normalized oracle cache

In [ ]:
rows = []
page_key = corpus_ids.astype(str)
for query_index, pages in enumerate(candidate_indices):
    pages = np.asarray(pages, dtype=int)
    branch_values = {
        "bm25_score": minmax(bm25_scores[query_index, pages]),
        "dense_score": minmax(dense_scores[query_index, pages]),
        "stage1_score": minmax(stage1_scores[query_index, pages]),
        "visual_score": minmax(visual_scores[query_index, pages]),
    }
    branch_ranks = {}
    for branch, values in branch_values.items():
        order = np.lexsort((page_key[pages], -values))
        ranks = np.empty(len(pages), dtype=int)
        ranks[order] = np.arange(1, len(pages) + 1)
        branch_ranks[branch] = ranks
    for local_index, page_index in enumerate(pages):
        rows.append({
            "dataset": DATASET,
            "query_id": str(query_ids[query_index]),
            "page_id": str(corpus_ids[page_index]),
            "relevance": qrel_lookup.get((query_ids[query_index], corpus_ids[page_index]), 0.0),
            "bm25_score": float(branch_values["bm25_score"][local_index]),
            "dense_score": float(branch_values["dense_score"][local_index]),
            "stage1_score": float(branch_values["stage1_score"][local_index]),
            "visual_score": float(branch_values["visual_score"][local_index]),
            "branch_ranks": json.dumps({
                "bm25": int(branch_ranks["bm25_score"][local_index]),
                "dense": int(branch_ranks["dense_score"][local_index]),
                "stage1": int(branch_ranks["stage1_score"][local_index]),
                "visual": int(branch_ranks["visual_score"][local_index]),
            }, separators=(",", ":")),
            "source": query_sources[query_index],
        })

retrieval_scores = pd.DataFrame(rows).sort_values(
    ["dataset", "query_id", "page_id"], kind="stable"
).reset_index(drop=True)
score_path = OUTPUT_DIR / "retrieval_scores.parquet"
retrieval_scores.to_parquet(score_path, index=False)
score_hash = dataframe_sha256(retrieval_scores, ["dataset", "query_id", "page_id"])
assert retrieval_scores[["dataset", "query_id", "page_id"]].duplicated().sum() == 0
assert retrieval_scores[["bm25_score", "dense_score", "stage1_score", "visual_score"]].notna().all().all()
print(f"Oracle cache ready: {len(retrieval_scores):,} rows, content hash {score_hash}")


## 6. Global Fusion, QARF and QPAF oracles

In [ ]:
oracle_rows, oracle_summary, subgroup = run_qpaf_oracle(
    retrieval_scores, grids=("w7",), n_bootstrap=2_000
)
for row in oracle_rows:
    assert row["qarf_metrics"]["ndcg10"] + 1e-12 >= row["global_metrics"]["ndcg10"]
    assert row["qpaf_metrics"]["ndcg10"] + 1e-12 >= row["qarf_metrics"]["ndcg10"]

write_jsonl(oracle_rows, OUTPUT_DIR / "fusion_oracle_results.jsonl")
write_json(oracle_summary, OUTPUT_DIR / "fusion_oracle_summary.json")
subgroup.to_csv(OUTPUT_DIR / "fusion_oracle_subgroups.csv", index=False)
result_frame = pd.DataFrame([
    {
        "dataset": row["dataset"],
        "query_id": row["query_id"],
        "grid": row["grid"],
        "global_ndcg10": row["global_metrics"]["ndcg10"],
        "qarf_ndcg10": row["qarf_metrics"]["ndcg10"],
        "qpaf_ndcg10": row["qpaf_metrics"]["ndcg10"],
        "delta_qarf_vs_global": row["delta_qarf_vs_global"],
        "delta_qpaf_vs_qarf": row["delta_qpaf_vs_qarf"],
    }
    for row in oracle_rows
])
result_frame.to_parquet(OUTPUT_DIR / "fusion_oracle_query_summary.parquet", index=False)
plot_qpaf(result_frame, OUTPUT_DIR / "global_qarf_qpaf_oracle.png")

feasibility = assess_short_pilot(oracle_summary, coverage_report, grid="w7")
write_json(feasibility, OUTPUT_DIR / "pilot_feasibility.json")
(OUTPUT_DIR / "pilot_feasibility.md").write_text(
    feasibility_markdown(feasibility), encoding="utf-8"
)

previous_manifest = None
manifest_path = OUTPUT_DIR / "run_manifest.yaml"
if manifest_path.exists():
    previous_manifest = yaml.safe_load(manifest_path.read_text(encoding="utf-8"))
manifest = {
    "study": "vidore_v3_finance_en_short_oracle",
    "scope": "exploratory_feasibility_not_formal_go_no_go",
    "seed": SEED,
    "dataset": {"id": DATASET, "revision": DATASET_REVISION, "queries": 309, "pages": 2942},
    "models": {
        name: {"id": model_name, "revision": checkpoint_sha(model_name)}
        for name, model_name in MODEL_NAMES.items()
    },
    "weights": "w7",
    "bootstrap_resamples": 2_000,
    "candidate_pool": {
        "initial": {"dse": 200, "bm25": 100, "bge_m3": 100},
        "expanded": {"dse": 300, "bm25": 200, "bge_m3": 200},
        "expanded_once": expanded_once,
        "coverage": final_coverage,
    },
    "hardware": {
        "gpu": gpu.name,
        "vram_gb": round(gpu_gb, 2),
        "cuda": torch.version.cuda,
        "torch": torch.__version__,
    },
    "packages": {
        name: importlib.metadata.version(name)
        for name in ["vidore-benchmark", "colpali-engine", "transformers", "datasets"]
    },
    "batch_sizes": BATCH_SIZES,
    "artifacts": {
        "retrieval_score_content_sha256": score_hash,
        "oracle_result_sha256": hashlib.sha256(
            (OUTPUT_DIR / "fusion_oracle_results.jsonl").read_bytes()
        ).hexdigest(),
    },
}
if previous_manifest:
    manifest["reproducibility"] = {
        "previous_score_hash_matches": previous_manifest.get("artifacts", {}).get(
            "retrieval_score_content_sha256"
        ) == score_hash,
        "previous_oracle_hash_matches": previous_manifest.get("artifacts", {}).get(
            "oracle_result_sha256"
        ) == manifest["artifacts"]["oracle_result_sha256"],
    }
else:
    manifest["reproducibility"] = "first_run_rerun_not_yet_checked"
manifest_path.write_text(yaml.safe_dump(manifest, sort_keys=False), encoding="utf-8")

print(feasibility_markdown(feasibility))
print(f"All outputs written to {OUTPUT_DIR}")
